In [1]:
# Imports and Paths

from pathlib import Path
from time import perf_counter

import numpy as np
import torch
import sys


repository_root = Path.cwd().resolve()

if not (repository_root / "ramsey").is_dir():
    parent_candidate = repository_root.parent

    if (parent_candidate / "ramsey").is_dir():
        repository_root = parent_candidate
    else:
        raise RuntimeError(
            "Could not locate the RamseyNumber repository root."
        )

if str(repository_root) not in sys.path:
    sys.path.insert(
        1,
        str(repository_root),
    )

from ramsey import (
    RArchiveBatch,
    RArchiveBatchConfig,
    REnvironment,
    REnvironmentConfig,
    RGraph,
    RMonochromaticObjective,
    RProblem,
    RRandomConstruction,
    RSearch,
    RSQLiteArchive,
    RTabuMemory,
    RTabuMemoryConfig,
)
from ramsey.nn import (
    RNeuralPolicy,
    load_training_checkpoint,
)


repository_root = Path.cwd().resolve()

if not (repository_root / "ramsey").is_dir():
    parent_candidate = repository_root.parent

    if (parent_candidate / "ramsey").is_dir():
        repository_root = parent_candidate
    else:
        raise RuntimeError(
            "Could not locate the RamseyNumber repository root."
        )


CHECKPOINT_PATH = (
    repository_root
    / "checkpoints"
    / "ramsey_policy_iteration_000099.pt"
)

DATABASE_PATH = (
    repository_root
    / "data"
    / "ramsey_colorings.sqlite3"
)

# Graph, Randomness, and Device

RANDOM_SEED = 202_608_041
N_VERTICES = 43

rng = np.random.default_rng(
    RANDOM_SEED
)

torch.manual_seed(
    RANDOM_SEED
)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )

graph = RGraph(
    RProblem.r55(
        n_vertices=N_VERTICES,
    )
)

if torch.cuda.is_available():
    device = torch.device(
        "cuda",
        torch.cuda.current_device(),
    )
else:
    device = torch.device(
        "cpu"
    )

# Load the Frozen Phase 1 Policy

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        CHECKPOINT_PATH
    )

restored = load_training_checkpoint(
    CHECKPOINT_PATH,
    graph=graph,
    device=device,
    rng=rng,
)

network = restored.network

network.eval()

for parameter in network.parameters():
    parameter.requires_grad_(False)

print(
    "Checkpoint:",
    CHECKPOINT_PATH,
)

print(
    "Completed iteration:",
    restored.completed_iteration,
)

print(
    "Device:",
    device,
)

print(
    "Network frozen:",
    not any(
        parameter.requires_grad
        for parameter in network.parameters()
    ),
)

Checkpoint: C:\code\RamseyNumber\checkpoints\ramsey_policy_iteration_000099.pt
Completed iteration: 99
Device: cuda:0
Network frozen: True


In [2]:
#
# Batch Archive - Make 500 distinct sub 800 score colorings
#

# 1) Imports and batch configuration
import numpy as np
from time import perf_counter

from ramsey import (
    RArchiveBatch,
    RArchiveBatchConfig,
    RRandomConstruction,
    RSearch,
)
from ramsey.nn import RNeuralPolicy

ROLLOUT_STEPS = 128
EDGE_TABU_TENURE = 20
VISITED_STATE_WINDOW = 2_000

BATCH_SCORE_LIMIT = 799
BATCH_TARGET_COUNT = 500
BATCH_MAXIMUM_ATTEMPTS = 2_500
BATCH_STEPS = 128

# Change this when intentionally beginning a new population run.
BATCH_RANDOM_SEED = 202_608_041
BATCH_RUN_NAME = "checkpoint-policy-sub-800"

# 2) Build a dedicated batch search
archive = RSQLiteArchive(
    DATABASE_PATH
)

batch_rng = np.random.default_rng(
    BATCH_RANDOM_SEED
)

batch_construction = RRandomConstruction(
    batch_rng
)

batch_memory = RTabuMemory(
    number_of_edges=graph.number_of_edges,
    config=RTabuMemoryConfig(
        edge_tenure=EDGE_TABU_TENURE,
        visited_state_window=VISITED_STATE_WINDOW,
    ),
)

batch_environment = REnvironment(
    graph=graph,
    objective=RMonochromaticObjective(),
    memory=batch_memory,
    config=REnvironmentConfig(
        max_steps=BATCH_STEPS,
        use_aspiration=True,
    ),
)

batch_policy = RNeuralPolicy(
    network=network,
    device=device,
    greedy=False,
)

batch_search = RSearch(
    environment=batch_environment,
    policy=batch_policy,
)

archive_batch = RArchiveBatch(
    graph=graph,
    construction=batch_construction,
    search=batch_search,
    archive=archive,
)

# 3) Inspect the current pool
eligible_before = (
    archive.coloring_count_in_score_range(
        maximum_score=BATCH_SCORE_LIMIT,
        graph=graph,
    )
)

print(
    "Existing distinct sub-800 colorings:",
    eligible_before,
)

print(
    "Target:",
    BATCH_TARGET_COUNT,
)

print(
    "Still needed:",
    max(
        0,
        BATCH_TARGET_COUNT - eligible_before,
    ),
)

# 4) Progress reporter
def report_archive_attempt(attempt) -> None:
    accepted_new_coloring = (
        attempt.in_score_range
        and attempt.new_unique_coloring
    )

    periodic_report = (
        (attempt.attempt + 1) % 25 == 0
    )

    if accepted_new_coloring or periodic_report:
        print(
            f"Attempt {attempt.attempt + 1:5d} | "
            f"initial="
            f"{attempt.search_result.initial_score:4d} | "
            f"best="
            f"{attempt.search_result.best_score:4d} | "
            f"accepted={accepted_new_coloring!s:5s} | "
            f"pool={attempt.eligible_count:4d}"
        )

# 5) Populate the archive
batch_start = perf_counter()

batch_result = archive_batch.populate(
    RArchiveBatchConfig(
        run_name=BATCH_RUN_NAME,
        target_count=BATCH_TARGET_COUNT,
        maximum_attempts=BATCH_MAXIMUM_ATTEMPTS,
        maximum_score=BATCH_SCORE_LIMIT,
        start_iteration=archive.coloring_count(
            graph
        ),
        record_steps=False,

        # Strict sub-800 population:
        # discard results scoring 800 or higher.
        save_out_of_range=False,
    ),
    observer=report_archive_attempt,
)

batch_elapsed = (
    perf_counter()
    - batch_start
)

# 6) Final Report
print()
print("Archive population complete")
print("---------------------------")

print(
    "Attempts:",
    batch_result.attempts_completed,
)

print(
    "Initial eligible:",
    batch_result.initial_eligible_count,
)

print(
    "Final eligible:",
    batch_result.final_eligible_count,
)

print(
    "New eligible:",
    batch_result.new_eligible_colorings,
)

print(
    "Target reached:",
    batch_result.target_reached,
)

print(
    "Best generated:",
    batch_result.best_score,
)

print(
    "Elapsed:",
    f"{batch_elapsed:.1f} seconds",
)

if batch_result.attempts_completed:
    print(
        "Seconds per attempt:",
        f"{batch_elapsed / batch_result.attempts_completed:.3f}",
    )

Existing distinct sub-800 colorings: 500
Target: 500
Still needed: 0

Archive population complete
---------------------------
Attempts: 0
Initial eligible: 500
Final eligible: 500
New eligible: 0
Target reached: True
Best generated: None
Elapsed: 0.0 seconds


In [3]:
sub_800_count = archive.coloring_count_in_score_range(
    maximum_score=799,
    graph=graph,
)

sub_800_records = archive.colorings_in_score_range(
    maximum_score=799,
    graph=graph,
)

print("Distinct sub-800 colorings:", sub_800_count)
print("Best score:", sub_800_records[0].score)
print("Worst accepted score:", sub_800_records[-1].score)

archive.close()

Distinct sub-800 colorings: 500
Best score: 689
Worst accepted score: 799
